In [5]:
import os
import pandas as pd

def auditar_nodo_cero():
    # La ruta exacta que configuramos en el script de descarga
    ruta_archivo = 'Data/API_Historico/Tamaulipas_2017_V4_Completo/nodo_581_2017_v4.parquet'
    # ruta_archivo = 'Data/Puerto_Rico_v3_2017/Finales/filtrado/dataset_v3_filtrado_diurno_2017.parquet'
    ruta_archivo = 'Data/Tamaulipas/2024/crudos_api/nodo_0_2024_v4.parquet'
    
    print("=== AUDITORÍA DE CALIDAD ESPACIOTEMPORAL: NODO 581 (TAMAULIPAS) ===")
    
    # Verificación de existencia
    if not os.path.exists(ruta_archivo):
        print(f"\n❌ El archivo aún no existe en: {ruta_archivo}")
        print("Si acabas de lanzar la descarga, espera unos segundos a que termine el primer nodo y vuelve a ejecutar.")
        return

    # 1. Análisis de Archivo Físico
    peso_bytes = os.path.getsize(ruta_archivo)
    peso_mb = peso_bytes / (1024 * 1024)
    print(f"\n📄 Estado: Archivo localizado correctamente.")
    print(f"💾 Peso en disco (Parquet comprimido): {peso_mb:.2f} MB")

    # 2. Carga en Memoria RAM
    df = pd.read_parquet(ruta_archivo, engine='pyarrow')
    
    # 3. Análisis Estructural (Tensor)
    filas, columnas = df.shape
    print(f"\n📊 ESTRUCTURA DEL DATASET:")
    # Un año no bisiesto (como 2017) debe tener exactamente 8,760 horas
    estado_filas = "✅ (Correcto: Año completo de 8,760 horas)" if filas == 8760 else f"⚠️ (Atención: Esperadas 8760, obtenidas {filas})"
    print(f"   -> Filas (Pasos de tiempo):  {filas:,} {estado_filas}")
    print(f"   -> Columnas (Atributos):     {columnas}")
    
    # 4. Integridad de los Datos (Checkeo de Nulos)
    nulos_totales = df.isnull().sum().sum()
    estado_nulos = "✅ (Cero nulos, matriz densa perfecta)" if nulos_totales == 0 else f"⚠️ (Se encontraron {nulos_totales} valores nulos)"
    print(f"   -> Integridad de datos:      {estado_nulos}")

    # 5. Esquema de Variables (Tipos de datos)
    print("\n📋 DICCIONARIO DE VARIABLES (Tipos de Datos):")
    # Imprimimos las columnas en un formato limpio y alineado
    for col in df.columns:
        tipo = str(df[col].dtype)
        print(f"   - {col:<30} | {tipo:<10}")

    # 6. Muestra Real (Preview)
    print("\n" + "="*80)
    print("🔍 VISTA PREVIA: PRIMERAS 3 HORAS DE OBSERVACIÓN")
    print("="*80)
    
    # Configurar Pandas para no truncar las columnas en la consola
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    
    print(df.head(3).to_string(index=False))
    print("="*80 + "\n")



In [6]:
if __name__ == "__main__":
    auditar_nodo_cero()

=== AUDITORÍA DE CALIDAD ESPACIOTEMPORAL: NODO 581 (TAMAULIPAS) ===

📄 Estado: Archivo localizado correctamente.
💾 Peso en disco (Parquet comprimido): 0.30 MB

📊 ESTRUCTURA DEL DATASET:
   -> Filas (Pasos de tiempo):  8,760 ✅ (Correcto: Año completo de 8,760 horas)
   -> Columnas (Atributos):     31
   -> Integridad de datos:      ✅ (Cero nulos, matriz densa perfecta)

📋 DICCIONARIO DE VARIABLES (Tipos de Datos):
   - nodo_id                        | int64     
   - year                           | int64     
   - month                          | int64     
   - day                            | int64     
   - hour                           | int64     
   - minute                         | int64     
   - temperature                    | float64   
   - alpha                          | float64   
   - aerosol_optical_depth          | float64   
   - asymmetry                      | float64   
   - clearsky_dhi                   | int64     
   - clearsky_dni                   | int64 

In [ ]:
import os
import io
import requests
import pandas as pd
from dotenv import load_dotenv

def verificar_alineacion_malla(muestra_size=15):
    """
    Toma una muestra aleatoria de nuestra malla generada y la compara
    con las coordenadas reales devueltas por el satélite de NREL.
    """
    print("=== AUDITORÍA DE ALINEACIÓN SATELITAL (RASTER SHIFT) ===")
    
    # 1. Cargar configuración y datos locales
    load_dotenv()
    API_KEY = os.getenv('API_KEY')
    EMAIL_USUARIO = os.getenv('EMAIL_USUARIO')
    
    if not API_KEY:
        print("❌ Error: Verifica tu archivo .env, no se detectó API_KEY.")
        return

    ruta_csv = 'Data/Geometria/metadata_nodos_tamaulipas_final.csv'
    
    try:
        df_meta = pd.read_csv(ruta_csv)
    except FileNotFoundError:
        print(f"❌ Error: No se encontró el archivo {ruta_csv}")
        return

    # 2. Tomar una muestra aleatoria
    # random_state asegura que siempre tome los mismos si lo corres varias veces
    df_muestra = df_meta.sample(n=muestra_size, random_state=42).copy()
    
    print(f"-> Archivo maestro cargado ({len(df_meta)} nodos totales).")
    print(f"-> Iniciando prueba de precisión en {muestra_size} nodos aleatorios...\n")
    
    print(f"{'Nodo':<6} | {'Lat Local':<10} | {'Lat NREL':<10} | {'Δ Lat':<10} | {'Lon Local':<10} | {'Lon NREL':<10} | {'Δ Lon':<10}")
    print("-" * 85)
    
    url_base = "https://developer.nlr.gov/api/nsrdb/v2/solar/nsrdb-GOES-aggregated-v4-0-0-download.csv"
    
    errores_lat = []
    errores_lon = []

    # 3. Consultar la API para cada nodo de la muestra
    for _, row in df_muestra.iterrows():
        nodo_id = int(row['nodo_id'])
        lat_local = round(row['latitude'], 4)
        lon_local = round(row['longitude'], 4)
        
        parametros = {
            'api_key': API_KEY,
            'email': EMAIL_USUARIO,
            'wkt': f"POINT({lon_local} {lat_local})",
            'names': '2017',
            'interval': '60',
            'utc': 'true',
            'leap_day': 'false'
        }
        
        try:
            # Petición a la API
            response = requests.get(url_base, params=parametros, timeout=30)
            response.raise_for_status()
            
            # Extraer metadatos de NREL (Fila 0)
            df_meta_api = pd.read_csv(io.StringIO(response.text), nrows=1)
            lat_nrel = round(float(df_meta_api['Latitude'].iloc[0]), 4)
            lon_nrel = round(float(df_meta_api['Longitude'].iloc[0]), 4)
            
            # Calcular diferencias (Deltas) absolutas
            delta_lat = abs(lat_local - lat_nrel)
            delta_lon = abs(lon_local - lon_nrel)
            
            errores_lat.append(delta_lat)
            errores_lon.append(delta_lon)
            
            # Imprimir fila de comparación
            print(f"{nodo_id:<6} | {lat_local:<10.4f} | {lat_nrel:<10.4f} | {delta_lat:<10.4f} | {lon_local:<10.4f} | {lon_nrel:<10.4f} | {delta_lon:<10.4f}")
            
        except Exception as e:
            print(f"{nodo_id:<6} | ERROR DE RED AL CONSULTAR API: {e}")

    # 4. Veredicto Final
    print("-" * 85)
    max_error_lat = max(errores_lat) if errores_lat else 0
    max_error_lon = max(errores_lon) if errores_lon else 0
    
    print(f"\nResumen de Precisión:")
    print(f"Error Máximo en Latitud: {max_error_lat:.4f} grados")
    print(f"Error Máximo en Longitud: {max_error_lon:.4f} grados")
    
    # Tolerancia matemática rigurosa: 0.001 grados (~111 metros de diferencia, que es casi nada en una malla de 4km)
    if max_error_lat < 0.001 and max_error_lon < 0.001:
        print("\n✅ VEREDICTO: PERFECTO. La malla generada localmente está perfectamente alineada con los píxeles del satélite GOES.")
    else:
        print("\n⚠️ VEREDICTO: DESFASE DETECTADO. La cuadrícula tiene un 'Raster Shift' significativo que debe corregirse.")

if __name__ == "__main__":
    verificar_alineacion_malla()

### Mapas - Espaciales

Uso (Python)

Elige región y año; las salidas se guardan en `Results/<Región>/<año>/` (el PNG ya incluye variable y año).

```python
from Utils.mapas_espaciales import MapaEspacial

REGION, ANIO = 'Tamaulipas', 2024          # o 'Puerto_Rico'
tag = {'Tamaulipas': 'tamaulipas', 'Puerto_Rico': 'pr'}[REGION]
meta = {'Tamaulipas': 'Data/Tamaulipas/metadata_nodos_tamaulipas.csv',
        'Puerto_Rico': 'Data/Puerto_Rico/metadata_nodos_pr.csv'}[REGION]

m = MapaEspacial(
    dataset=f'Data/{REGION}/{ANIO}/Finales/completo/dataset_{tag}_completo_24h_{ANIO}.parquet',
    metadata=meta, dir_salida=f'Results/{REGION}/{ANIO}',
    anio=ANIO, region=REGION.replace('_', ' '))

m.mapa('temperature', titulo_var='Temperatura', unidades='°C', cmap='inferno')      # anual
m.mapa('ghi', filtro='mensual', mes=6, titulo_var='GHI', unidades='W/m²')           # junio
m.mapa('wind_speed', filtro='diario', dia=f'{ANIO}-06-21', unidades='m/s')          # un día
m.mapa('ghi', agregacion='suma_diaria', factor=1/1000,                              # radiación diaria
       titulo_var='Radiación solar media diaria', unidades='kWh/m²/día')
m.mapa_12_meses('temperature', titulo_var='Temperatura', unidades='°C', cmap='inferno')
```

Uso (CLI) — usa las rutas por defecto (Tamaulipas 2017). Para otra región/año, usa la API de Python de arriba.

```bash
python Utils/mapas_espaciales.py --variable temperature --unidades "°C" --cmap inferno
python Utils/mapas_espaciales.py --variable ghi --agregacion suma_diaria --factor 0.001 --unidades "kWh/m²/día"
python Utils/mapas_espaciales.py --variable wind_speed --filtro diario --dia 2017-06-21 --unidades m/s
python Utils/mapas_espaciales.py --variable temperature --panel12 --unidades "°C" --cmap inferno
```


In [13]:
from Utils.mapas_espaciales import MapaEspacial

# --- Configuración: elige región y año ---
REGION = 'Puerto_Rico'        # 'Tamaulipas' o 'Puerto_Rico'
ANIO   = 2024

# Rutas por región (dataset consolidado + metadata). Las salidas se guardan en
# Results/<Región>/<año>/ (el nombre del PNG ya incluye la variable y el año).
_CFG = {
    'Tamaulipas':  {'tag': 'tamaulipas', 'metadata': 'Data/Tamaulipas/metadata_nodos_tamaulipas.csv', 'titulo': 'Tamaulipas'},
    'Puerto_Rico': {'tag': 'pr',         'metadata': 'Data/Puerto_Rico/metadata_nodos_pr.csv',         'titulo': 'Puerto Rico'},
}
cfg = _CFG[REGION]
dataset    = f'Data/{REGION}/{ANIO}/Finales/completo/dataset_{cfg["tag"]}_completo_24h_{ANIO}.parquet'
dir_salida = f'Results/{REGION}/{ANIO}'

m = MapaEspacial(dataset=dataset, metadata=cfg['metadata'],
                 dir_salida=dir_salida, anio=ANIO, region=cfg['titulo'])

m.mapa_12_meses('ghi', agregacion='suma_diaria', factor=1/1000,                              # radiación diaria
       titulo_var='Radiación solar media diaria', unidades='kWh/m²/día')
m.mapa('ghi', agregacion='suma_diaria', factor=1/1000,                              # radiación diaria
       titulo_var='Radiación solar media diaria', unidades='kWh/m²/día')
m.mapa_12_meses('temperature', titulo_var='Temperatura', unidades='°C')
m.mapa('temperature', titulo_var='Temperatura', unidades='°C')
m.mapa_12_meses('relative_humidity', titulo_var='Humedad relativa', unidades='%')
m.mapa('relative_humidity', titulo_var='Humedad relativa', unidades='%')


=== PANEL 12 MESES Radiación solar media diaria (media diaria) ===
Escala compartida: [3.07, 7.12] kWh/m²/día
✅ 5.5 s -> Results/Puerto_Rico/2024/mapa_ghi_2024_12meses.png

=== MAPA Radiación solar media diaria (media diaria) | Anual ===
Nodos: 754 | valor medio 5.45 | rango [4.66, 6.02] kWh/m²/día
✅ 1.2 s -> Results/Puerto_Rico/2024/mapa_ghi_2024_anual.png

=== PANEL 12 MESES Temperatura (media) ===
Escala compartida: [20.69, 30.73] °C
✅ 4.7 s -> Results/Puerto_Rico/2024/mapa_temperature_2024_12meses.png

=== MAPA Temperatura (media) | Anual ===
Nodos: 754 | valor medio 27.90 | rango [22.53, 29.23] °C
✅ 1.1 s -> Results/Puerto_Rico/2024/mapa_temperature_2024_anual.png

=== PANEL 12 MESES Humedad relativa (media) ===
Escala compartida: [70.25, 98.25] %
✅ 4.7 s -> Results/Puerto_Rico/2024/mapa_relative_humidity_2024_12meses.png

=== MAPA Humedad relativa (media) | Anual ===
Nodos: 754 | valor medio 81.16 | rango [75.20, 94.40] %
✅ 1.1 s -> Results/Puerto_Rico/2024/mapa_relative_humidity

'Results/Puerto_Rico/2024/mapa_relative_humidity_2024_anual.png'

### Selección interactiva de nodos a eliminar (lasso)

Requiere `%matplotlib widget` (paquete `ipympl`). Arrastra sobre el mapa para encerrar nodos: **botón izquierdo agrega**, **botón derecho quita**. Empieza con los **58 nodos sobre agua** (`msnm==0`) ya marcados en negro; ajústalos a mano.

Al terminar, en otra celda: `sel.guardar()` exporta a `Data/Tamaulipas/nodos_a_eliminar.csv`.

In [ ]:
%matplotlib widget
from Utils.seleccion_lasso import SelectorNodos

# Arranca con los 58 nodos sobre agua (msnm==0) preseleccionados (en negro).
#   · Arrastra botón IZQUIERDO -> AGREGA nodos a la selección
#   · Arrastra botón DERECHO   -> QUITA nodos de la selección
#   · sel.reiniciar()          -> vacía toda la selección
sel = SelectorNodos(preseleccion_msnm=0.0)

# --- cuando termines de ajustar, ejecuta en OTRA celda: ---
# sel.guardar()        # -> Data/Tamaulipas/nodos_a_eliminar.csv
# sel.seleccionados    # lista de nodo_id seleccionados

In [ ]:
sel.guardar()        # -> Data/Tamaulipas/nodos_a_eliminar.csv
sel.seleccionados    # lista de nodo_id seleccionados

### Revisión interactiva de los nodos de agua restantes

Mapa con **zoom** e imagen **satelital** para inspeccionar los 4 nodos `msnm==0` que quedaron tras el filtro. Haz clic en los marcadores rojos para ver su `nodo_id`/`nodo_id_4501`. No requiere `%matplotlib widget` (folium se renderiza solo).

In [ ]:
from Utils.mapa_interactivo import mapa_nodos

# Mapa con zoom + imagen satelital (Esri) para revisar si están sobre agua.
# Rojo = nodos a revisar (clic para ver nodo_id / msnm) · Azul = vecinos (contexto).
# Estos son los 4 nodos con msnm==0 que quedaron tras el filtro de 117:
mapa_nodos([2603, 2604, 2650, 2701])

In [11]:
import h5py
import pandas as pd

ruta = "Data/datos_nsrdb/nsrdb_puerto_rico_1998.h5"   # ajusta a tu ruta local

with h5py.File(ruta, "r") as h5:
    meta = pd.DataFrame(h5["meta"][...])
    print("Número de nodos:", meta.shape[0])
    print("Variables (datasets):", list(h5.keys()))
    print(meta.head())

with h5py.File(ruta, "r") as h5:
    print("Shape de 'ghi' (tiempo, nodos):", h5["ghi"].shape)
    print("time_index:", h5["time_index"].shape)

for col in meta.select_dtypes(include="object").columns:
    meta[col] = meta[col].str.decode("utf-8")

Número de nodos: 2480
Variables (datasets): ['air_temperature', 'clearsky_dhi', 'clearsky_dni', 'clearsky_ghi', 'coordinates', 'dhi', 'dni', 'ghi', 'meta', 'solar_zenith_angle', 'surface_albedo', 'surface_pressure', 'time_index', 'total_precipitable_water', 'wind_speed']
    latitude  longitude  elevation
0  18.120001 -67.930000       56.0
1  18.100000 -67.930000       63.0
2  18.080000 -67.930000       37.0
3  18.059999 -67.930000        8.0
4  18.120001 -67.910004       71.0
Shape de 'ghi' (tiempo, nodos): (105120, 2480)
time_index: (105120,)
